> ## SOLUTION / ANSWER KEY &mdash; Lab 6.11
> This is the **completed** notebook (all `___` blanks filled). For the student version, open
> [`../lab-11-guardrails-observability.ipynb`](../lab-11-guardrails-observability.ipynb). Trainer use &mdash; or self-check after you've tried it yourself.

# Lab 6.11 &mdash; Guardrails & Observability

**Level:** Advanced &nbsp;|&nbsp; **Est. time:** 35 min &nbsp;|&nbsp; **Day 3 &middot; Module 6 &mdash; Frameworks for Building AI Agents**

### What you'll do
- See why a raising tool aborts the whole agent run -- and fix it
- Allow-list the tools the agent may call + validate input
- Attach a REAL BaseCallbackHandler and read every tool event from a live run

> **How this lab works (near-real):** you have a local Ollama running `llama3.1:8b`. Read the **Concept**, fill the real `___` blanks in **Build it** (real tool bodies, real prompts, the real `create_agent` call), then **Run it for real** &mdash; a real LLM drives a real agent over real tools &mdash; and **read the trace** to see exactly what it did. Finish with an open **Your turn**. There is **no auto-grader**; the goal is a working agent and a trace you can read.

> **Framework note:** these labs use the **real** LangChain (`langchain`, `langchain-core`, `langchain-ollama`, `langgraph`) and a **real local model** (`ollama run llama3.1:8b`, pinned to `http://127.0.0.1:11434`). If Ollama is down, the run cells print how to start it instead of crashing. A `@tool` must **catch its own errors and return a string** &mdash; a tool that *raises* aborts the whole agent run (you'll see exactly this in Lab 11).

**Reference:** [Module 6 slides &mdash; Observability & debugging](../../presentation/day3-module6-frameworks-for-building-ai-agents.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html) &nbsp;&middot;&nbsp; [All Module 6 labs](./index.html)

In [ ]:
# Setup -- run me first
import os, socket, pathlib
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True))   # SERPER_API_KEY, WOLFRAM_ALPHA_APPID, GROQ/OPENAI keys

WORK = "/tmp/biaa-lab-06-11"
os.makedirs(WORK, exist_ok=True)

def ollama_up(host="127.0.0.1", port=11434):
    """True if a local Ollama server is listening. If it's down, start it with:  ollama run llama3.1:8b"""
    try:
        with socket.create_connection((host, port), timeout=1):
            return True
    except OSError:
        return False

from langchain_ollama import ChatOllama
# Day-3 provider: a REAL local model. Pin the host -- plain 'localhost' can give 'No route to host'.
llm = ChatOllama(model="llama3.1:8b", temperature=0, base_url="http://127.0.0.1:11434")

def print_trace(result):
    """Print a REAL agent message trace: tool calls the model made, tool observations, final answer."""
    for m in result["messages"]:
        for tc in (getattr(m, "tool_calls", None) or []):
            print("TOOL CALL:", tc["name"], tc["args"])
        if type(m).__name__ == "ToolMessage":
            print("OBS:", str(m.content)[:200])
        elif str(getattr(m, "content", "")).strip():
            print(type(m).__name__, ":", str(m.content)[:300])

if ollama_up():
    print("Ollama reachable at 127.0.0.1:11434 | model:", llm.model, "| WORK:", WORK)
else:
    print("Ollama NOT reachable -- start it with:  ollama run llama3.1:8b")
    print("(The 'Run it for real' cells will print this note instead of crashing.)  WORK:", WORK)

## Concept
Autonomy needs **guardrails** and **observability** (deck slides 15, 17). Two lessons here, both real:
**(1) A tool must never raise.** If a `@tool` throws on a bad argument, the exception can abort the entire
agent run &mdash; so tools **catch and return a string**. **(2) Observe everything.** LangChain exposes a
real **`BaseCallbackHandler`**; attach one and it fires on every tool start/end so you can trace and audit
(LangSmith / Langfuse capture full runs this way). You also keep a deterministic **allow-list** + input
**validation**.

In [ ]:
from langchain_core.tools import tool

# The WRONG way: this tool raises on bad input.
@tool
def unsafe_divide(expr: str) -> str:
    """Divide two numbers like '10/2'. (Unsafe demo -- raises on bad input!)"""
    a, b = expr.split("/")
    return str(int(a) / int(b))     # ZeroDivisionError / ValueError propagate -> can abort the agent

print("ok  :", unsafe_divide.invoke("10/2"))
try:
    print(unsafe_divide.invoke("10/0"))
except Exception as e:
    print("raised:", type(e).__name__, "-- inside an agent this can abort the whole run")

## Build it
Write the **safe** version of the tool (catch + return a string), the allow-list check, the validator,
and a **real** callback handler that records each tool result.

In [ ]:
import ast, operator
# safe arithmetic: walk a parsed AST of numbers + (+ - * / ** unary-minus) -- never bare eval()
_OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
        ast.Div: operator.truediv, ast.USub: operator.neg, ast.Pow: operator.pow}
def safe_calc(expr):
    def ev(node):
        if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
            return node.value
        if isinstance(node, ast.BinOp):
            return _OPS[type(node.op)](ev(node.left), ev(node.right))
        if isinstance(node, ast.UnaryOp):
            return _OPS[type(node.op)](ev(node.operand))
        raise ValueError("unsupported expression")
    return ev(ast.parse(expr, mode="eval").body)

from langchain_core.tools import tool
from langchain_core.callbacks import BaseCallbackHandler

@tool
def calculator(expression: str) -> str:
    """Do exact arithmetic on an expression such as 10/2."""
    try:
        return str(safe_calc(expression))
    except Exception:
        return "error: not a numeric expression"

ALLOWED = {"calculator", "lookup"}

def is_allowed(action):
    return action in ALLOWED

def validate(expr):
    ok_chars = set("0123456789+-*/(). ")
    return all(c in ok_chars for c in expr) and any(c.isdigit() for c in expr)

class TraceHandler(BaseCallbackHandler):
    """A REAL LangChain callback: LangChain calls these methods on every tool event."""
    def __init__(self):
        self.events = []
    def on_tool_end(self, output, **kw):
        self.events.append(("tool_end", str(output)[:80]))

try:
    print("safe calculator(10/0):", calculator.invoke("10/0"))   # returns a string, no crash
    print("is_allowed(calculator):", is_allowed("calculator"), "| is_allowed(delete_db):", is_allowed("delete_db"))
    print("validate('2+2'):", validate("2+2"), "| validate('__import__'):", validate("__import__"))
except Exception as e:
    print("(Fill the ___ blanks above, then re-run.)", type(e).__name__)

## Run it for real &amp; read the trace
Attach your REAL callback to a real agent run via `config={'callbacks': [...]}` and read the tool events it captured.

_This calls the real `llama3.1:8b` via Ollama. If Ollama is down the cell prints how to start it instead of crashing._

In [ ]:
if not ollama_up():
    print("Ollama not reachable -- start `ollama run llama3.1:8b`, then re-run this cell.")
else:
    try:
        from langchain.agents import create_agent
        handler = TraceHandler()
        agent = create_agent(llm, tools=[calculator])
        result = agent.invoke({"messages": [("user", "What is 144 divided by 12?")]},
                              config={"recursion_limit": 6, "callbacks": [handler]})
        print("final:", result["messages"][-1].content)
        print("callback captured tool events:")
        for e in handler.events:
            print("  ", e)
    except Exception as e:
        print("(Fill the ___ blanks above, then re-run.)", type(e).__name__)

## What to notice
- The safe `calculator` returned a **string** for `10/0` &mdash; inside the live run that's the difference between a recoverable step and an aborted agent.
- Your `TraceHandler` is a **real** `BaseCallbackHandler`; LangChain called `on_tool_end` for you on every tool result. That's the same hook LangSmith / Langfuse use.
- Allow-list + validation + `recursion_limit` + a callback = an agent you can trust and audit.

## Your turn (open task &mdash; no grader)
Bind the **unsafe** tool (`unsafe_divide`) to the agent instead and ask *"What is 10 divided by 0?"* &mdash;
watch how a raising tool derails the run. Then switch back to the safe `calculator`. Add an
`on_tool_start(self, serialized, input_str, **kw)` method to your handler to log inputs too. **What good
looks like:** the safe agent recovers and answers; your callback logs every tool start and end.

In [ ]:
# --- Reference answer (ONE good way to do the 'Your turn' task -- compare with your own) ---
from langchain.agents import create_agent

class FullTraceHandler(TraceHandler):
    def on_tool_start(self, serialized, input_str, **kw):
        self.events.append(("tool_start", str(input_str)[:80]))   # log inputs too

if ollama_up():
    handler = FullTraceHandler()
    unsafe_agent = create_agent(llm, tools=[unsafe_divide])   # the raising tool
    try:
        r = unsafe_agent.invoke({"messages": [("user", "What is 10 divided by 0?")]},
                                config={"recursion_limit": 6, "callbacks": [handler]})
        print("unsafe final:", r["messages"][-1].content)
    except Exception as e:
        print("the raising tool derailed the run:", type(e).__name__)
    print("events:", handler.events)
    safe_agent = create_agent(llm, tools=[calculator])         # switch back to the safe tool
    print("safe final:", safe_agent.invoke({"messages": [("user", "What is 10 divided by 0?")]},
                                            config={"recursion_limit": 6})["messages"][-1].content)
else:
    print("(start Ollama: ollama run llama3.1:8b)")

---
### Key takeaway
Tools that return strings (never raise) + allow-list + validation + a real callback turn an autonomous agent from a liability into something you can trust and debug. Day 5 goes deeper.

[&#8592; All Module 6 labs](./index.html) &nbsp;&middot;&nbsp; [Module 6 slides](../../presentation/day3-module6-frameworks-for-building-ai-agents.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html)

<sub>&copy; 2026 Gheware DevOps &amp; Agentic AI &middot; Building Intelligent AI Agents &middot; devops.gheware.com &middot; Trainer: Rajesh Gheware</sub>